# 🏥 YumiCare — Genuine Fetal Ultrasound GAN + Segmentation Training
### Real GPU Training | No Fake Fallback | Class-Balanced Pipeline

**Hardware Target:** Google Colab GPU (T4 / A100 recommended)  

---

### 📋 Pipeline Setup
1. **Github Repo:** Code is automatically cloned from your public Github (`itzzSPcoder/YumiCare`).
2. **Dataset:** Since datasets are large, you must upload the local `Dataset` folder directly to your Google Drive root (so it appears as `My Drive/Dataset`). The notebook will automatically link it.

In [ ]:
# ============================================================
# CELL 1: Install dependencies & verify GPU
# ============================================================
!pip install -q openpyxl pandas scipy pillow matplotlib tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import pandas as pd
import random, os, time, collections, sys
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — training will be very slow on CPU!')

In [ ]:
# ============================================================
# CELL 2: Clone Repo & Link Dataset from Google Drive
# ============================================================
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Clone Public Repository
REPO_URL = 'https://github.com/itzzSPcoder/YumiCare.git'
WORK_DIR = Path('/content/YumiCare')

if not WORK_DIR.exists():
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull

# 3. Link Dataset from Google Drive
# Ensure you have uploaded the 'Dataset' folder to your Google Drive root!
DRIVE_DATASET = Path('/content/drive/MyDrive/Dataset')
LOCAL_DATASET = WORK_DIR / 'Dataset'

if not DRIVE_DATASET.exists():
    raise FileNotFoundError("❌ Could not find 'Dataset' folder in your Google Drive. Please upload it to your Google Drive root first.")

if not LOCAL_DATASET.exists():
    print("✅ Linking Dataset from Google Drive...")
    os.symlink(DRIVE_DATASET, LOCAL_DATASET)

# Set Working Directory to scripts
sys.path.insert(0, str(WORK_DIR / 'scripts'))
os.chdir(WORK_DIR / 'scripts')
print(f'✅ Workspace Ready! Current Dir: {os.getcwd()}')

In [ ]:
# ============================================================
# CELL 3: Dataset Analysis — Verify Imbalance
# ============================================================
RUNS_DIR = WORK_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect dataset path based on how it was uploaded to Google Drive
DATASET_DIR = WORK_DIR / 'Dataset' / '8265464'
if not DATASET_DIR.exists() or not list(DATASET_DIR.glob('*')):
    DATASET_DIR = WORK_DIR / 'Dataset'  # In case 8265464 contents were uploaded directly into Dataset

print(f'🔎 Using Dataset Path: {DATASET_DIR}')

# Count images per scan type
view_counts = {}
for subgroup in DATASET_DIR.glob('*'):
    if not subgroup.is_dir(): continue
    seg = next(subgroup.glob('*-Segmentation'), None)
    if not seg: continue
    mask_dir = seg / 'SegmentationClass'
    if mask_dir.exists():
        view_counts[subgroup.name] = len(list(mask_dir.glob('*.png')))

if not view_counts:
    raise ValueError('\n❌ NO MASKS FOUND! Please check your Google Drive:\n1. Make sure you uploaded the actual subfolders (like Abdomen-Segmentation).\n2. Make sure the masks are inside the SegmentationClass folders.')

print('📊 View-level image counts:')
for k, v in sorted(view_counts.items(), key=lambda x: -x[1]):
    print(f'   {k}: {v} images')

# Pixel-level class distribution (sample 100 masks)
import collections
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

pixel_counts = collections.Counter()
sampled = 0
for subgroup in DATASET_DIR.glob('*'):
    seg = next(subgroup.glob('*-Segmentation'), None)
    if not seg: continue
    mask_dir = seg / 'SegmentationClass'
    if not mask_dir.exists(): continue
    for mp in list(mask_dir.glob('*.png'))[:25]:
        arr = np.array(Image.open(mp).convert('RGB'))
        pixel_counts['Background'] += int((arr.sum(-1) == 0).sum())
        pixel_counts['Brain (Red)'] += int(np.all(arr == [255,0,0], axis=-1).sum())
        pixel_counts['CSP (Green)'] += int(np.all(arr == [0,255,0], axis=-1).sum())
        pixel_counts['LV (Blue)']   += int(np.all(arr == [0,0,255], axis=-1).sum())
        sampled += 1

total_px = sum(pixel_counts.values())
if total_px == 0:
    raise ValueError('No pixels found! Dataset seems empty or masks are missing.')

print(f'\n📊 Pixel-level class distribution ({sampled} masks sampled):')
for cls, count in pixel_counts.items():
    print(f'   {cls}: {count:,} px ({100*count/total_px:.3f}%)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(view_counts.keys(), view_counts.values(), color=['#2196F3','#4CAF50','#FF9800','#E91E63'])
axes[0].set_title('View-Level Image Count', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylabel('# Images')

colors = ['#607D8B', '#F44336', '#4CAF50', '#2196F3']
wedges, texts, autotexts = axes[1].pie(
    pixel_counts.values(), labels=pixel_counts.keys(),
    colors=colors, autopct='%1.3f%%', startangle=90
)
axes[1].set_title('Pixel-Level Class Distribution\n(Note: CSP + LV < 0.15%!)', fontweight='bold')
plt.tight_layout()
plt.savefig(RUNS_DIR / 'dataset_imbalance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset analysis complete')


In [ ]:
# ============================================================
# CELL 4: Load Genuine Training Modules
# ============================================================
from fetal_gan_architecture import (
    UltrasoundMaskConditionedGenerator,
    UltrasoundPatchCritic,
    compute_gradient_penalty,
    compute_class_weights,
    DiceLoss,
)
from train_fetal_gan import (
    FetalUltrasoundGANDataset,
    synthesize_synthetic_anatomical_mask,
    train_fetal_gan,
    generate_synthetic_dataset,
    save_gan_synthesis_preview,
)
from run_segmentation_benchmark import (
    run_benchmark,
    create_premium_excel_report,
    save_visualization_plots,
    DATASET_PIXEL_COUNTS,
)
print('✅ All modules loaded')

In [ ]:
# ============================================================
# CELL 5: Hyperparameter Configuration
# ============================================================

# ── GAN Training ──────────────────────────────────────────
GAN_EPOCHS      = 50       # Colab T4: ~45 min | A100: ~15 min
GAN_BATCH_SIZE  = 8
GAN_LR          = 1e-4
GAN_LAMBDA_GP   = 10.0     # WGAN-GP penalty weight
GAN_LAMBDA_L1   = 50.0     # Pixel reconstruction term
GAN_CRITIC_ITER = 5        # Critic steps per generator step
GAN_GENERATE    = 500      # Synthetic pairs to produce
GAN_MINORITY_R  = 0.40     # 40% generated with enlarged CSP/LV

# ── Segmentation Training ─────────────────────────────────
SEG_EPOCHS      = 15       # Per model per regime
SEG_BATCH_SIZE  = 8
SEG_LR          = 1e-4
IMAGE_SIZE      = 128

# ── Paths ────────────────────────────────────────────────
SYN_DIR  = WORK_DIR / 'Dataset' / 'synthetic_gan'

print('📋 Configuration:')
print(f'   GAN: {GAN_EPOCHS} epochs | batch={GAN_BATCH_SIZE} | generate={GAN_GENERATE} pairs')
print(f'   Seg: {SEG_EPOCHS} epochs × 5 models × 3 regimes = {SEG_EPOCHS*5*3} total training runs')
print(f'   Output: {RUNS_DIR}')

In [ ]:
# ============================================================
# CELL 6: Train cWGAN-GP — Genuine GAN Training
# ============================================================
print('🚀 Starting genuine cWGAN-GP training...')
print(f'   WeightedRandomSampler: CSP/LV-rich samples oversampled up to 20x')

gan_start = time.time()
generator, critic = train_fetal_gan(
    dataset_dir   = DATASET_DIR,
    output_dir    = RUNS_DIR,
    epochs        = GAN_EPOCHS,
    batch_size    = GAN_BATCH_SIZE,
    lr            = GAN_LR,
    lambda_gp     = GAN_LAMBDA_GP,
    lambda_l1     = GAN_LAMBDA_L1,
    critic_iters  = GAN_CRITIC_ITER,
    dry_run       = False,
)
gan_time = time.time() - gan_start
print(f'\n✅ GAN training complete in {gan_time/60:.1f} minutes')

In [ ]:
# ============================================================
# CELL 7: Generate Synthetic Dataset (class-balanced)
# ============================================================
print(f'🎨 Generating {GAN_GENERATE} synthetic ultrasound pairs...')

gen_stats = generate_synthetic_dataset(
    generator      = generator,
    output_dir     = SYN_DIR,
    num_samples    = GAN_GENERATE,
    image_size     = IMAGE_SIZE,
    minority_ratio = GAN_MINORITY_R,
)

save_gan_synthesis_preview(generator, RUNS_DIR / 'gan_synthesis_preview.png', 6, IMAGE_SIZE)

from IPython.display import Image as IPImage
IPImage(str(RUNS_DIR / 'gan_synthesis_preview.png'), width=700)

In [ ]:
# ============================================================
# CELL 8: Genuine Segmentation Benchmark
# ============================================================
print('🚀 Starting genuine segmentation benchmark...')
seg_start = time.time()

results_df, per_class_df, ablation_df = run_benchmark(
    dataset_dir    = DATASET_DIR,
    output_dir     = RUNS_DIR,
    augmentation_mode = 'all',
    seg_epochs     = SEG_EPOCHS,
    seg_batch_size = SEG_BATCH_SIZE,
    seg_lr         = SEG_LR,
)

seg_time = time.time() - seg_start
print(f'\n✅ Segmentation benchmark complete in {seg_time/60:.1f} minutes')

In [ ]:
# ============================================================
# CELL 9: Display & Save Genuine Results
# ============================================================
print('📊 GENUINE BENCHMARK RESULTS:')
display_cols = ['model', 'mean_iou', 'mean_dice', 'precision', 'recall', 'latency_ms']
print(results_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))

genuine_csv = RUNS_DIR / 'genuine_benchmark_results.csv'
results_df.to_csv(genuine_csv, index=False)
ablation_df.to_csv(RUNS_DIR / 'genuine_ablation_results.csv', index=False)
per_class_df.to_csv(RUNS_DIR / 'genuine_per_class_results.csv', index=False)
print(f'\n✅ Results saved to: {genuine_csv}')

In [ ]:
# ============================================================
# CELL 10: Export Premium Excel Report
# ============================================================
excel_path = RUNS_DIR / 'genuine_fetal_segmentation_report.xlsx'
create_premium_excel_report(results_df, per_class_df, ablation_df, excel_path)
print(f'✅ Excel report saved: {excel_path}')

In [ ]:
# ============================================================
# CELL 11: Download Results to Local Machine
# ============================================================
from google.colab import files
import zipfile

zip_path = RUNS_DIR / 'genuine_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in [
        'genuine_benchmark_results.csv',
        'genuine_ablation_results.csv',
        'genuine_per_class_results.csv',
        'genuine_fetal_segmentation_report.xlsx',
        'gan_synthesis_preview.png',
        'dataset_imbalance_analysis.png',
    ]:
        fp = RUNS_DIR / f
        if fp.exists():
            zf.write(fp, f)
            print(f'  Added: {f}')

print(f'\n📦 Downloading results zip...')
files.download(str(zip_path))